In [1]:
# Import Libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.datasets import FashionMNIST
from torchvision import transforms
from torch.utils.data import DataLoader, Subset, random_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix)

import numpy as np
import pandas as pd

## Loading and splitting data

In [2]:
# Set random seeds
torch.manual_seed(123)
np.random.seed(123)

# Loading the data.
transform = transforms.Compose([
 transforms.ToTensor(),
 transforms.Normalize((0.5,), (0.5,))])

fashion_data = FashionMNIST(
    root = './data',
    download = True,
    train = True,
    transform = transform)

indices = torch.randperm(len(fashion_data))[:12000]

fashion_subset = Subset(fashion_data, indices)

# Splitting test and train data
train_size = int(0.8 * len(fashion_subset))
test_size = len(fashion_subset) - train_size

train_dataset, test_dataset = random_split(
    fashion_subset,
    [train_size, test_size])

# Dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size = 64,
    shuffle = True)

test_loader = DataLoader(
    test_dataset,
    batch_size = 64,
    shuffle = False)

## MLP Model

In [3]:
class MLP(nn.Module):
    
    def __init__(self, input_size, hidden_size, output_size):
        super(MLP, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, output_size)
        self.activation = nn.ReLU()

    def forward(self, x):
        # Forward pass through the network
        x = self.activation(self.fc1(x))
        x = self.fc2(x)
        return x

## Training and tuning hyperparameter (Hidden layer size)

In [4]:
# Get input and output sizes automatically
sample_image, _ = train_dataset[0]

input_size = sample_image.numel()   # 28*28 = 784
output_size = len(fashion_data.classes)

hidden_sizes = [32, 64, 128]

epochs = 15

best_acc = 0
best_model = None
best_loss = None

acc_32 = 0
model_32 = None
loss_32 = None

acc_64 = 0
model_64 = None
loss_64 = None

acc_128 = 0
model_128 = None
loss_128 = None

# Testing range of hidden layer sizes (Hyperparameter tuning)
for h in hidden_sizes:

    model = MLP(input_size, h, output_size)

    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    # Training
    for epoch in range(epochs):

        model.train()

        for images, labels in train_loader:

            images = images.view(images.size(0), -1)

            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # Evaluate training accuracy
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in train_loader:

            images = images.view(images.size(0), -1)

            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    acc = correct / total *100

    print(f"Hidden size {h}: Accuracy = {acc:.2f}%")

    # Saving trained models withing different hidden layer sizes
    if h == 32:
        acc_32 = acc
        model_32 = model
        loss_32 = loss.item()
    
    if h == 64:
        acc_64 = acc
        model_64 = model
        loss_64 = loss.item()

    if h == 128:
        acc_128 = acc
        model_128 = model
        loss_128 = loss.item()

Hidden size 32: Accuracy = 89.40%
Hidden size 64: Accuracy = 90.99%
Hidden size 128: Accuracy = 92.70%


In [5]:
model = model_32

print(f"\nBest hidden size: 32"
    f"\nAccuracy: {acc_32:.2f}%"
    f"\nFinal Loss: {loss_32:.2f}%")


Best hidden size: 32
Accuracy: 89.40%
Final Loss: 0.35%


The hyperparameter selected for tuning was the number of neurons in the hidden layer. This hyperparameter was chosen because it influences the model's ability to learn patterns from the data. A hidden layer with too few neurons may lead to underfitting, where the model is unable to capture important relationships in the dataset. Conversely, a hidden layer with too many neurons can increase training time and may result in overfitting. Several hidden layer sizes were therefore evaluated. A hidden layer size of 32 neurons was selected because it achieved a high level of accuracy while using fewer parameters than the larger alternatives. Although 64 neurons produced slightly higher accuracy, the improvement was marginal. Larger hidden layers, such as 128 neurons, may have increased the risk of overfitting without providing a substantial performance benefit.

## Evaluating

In [6]:
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:

        images = images.view(images.size(0), -1)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = (np.array(all_preds) == np.array(all_labels)).mean() * 100
    
precision = precision_score(all_labels, all_preds, average='macro') * 100
recall = recall_score(all_labels, all_preds, average='macro') * 100
f1 = f1_score(all_labels, all_preds, average='macro') * 100
    
print(f"Accuracy : {accuracy:.2f}%")
print(f"Precision: {precision:.2f}%")
print(f"Recall   : {recall:.2f}%")
print(f"F1 Score : {f1:.2f}%")

Accuracy : 85.29%
Precision: 85.45%
Recall   : 85.14%
F1 Score : 85.16%


The model achieved an accuracy of 85.29%, exceeding the target accuracy of 80%.

## Correlation matrix

In [7]:
def evaluate_model(model, test_loader):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.view(inputs.size(0), -1)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.tolist())
    return all_preds, all_labels

preds, labels = evaluate_model(model, test_loader)
conf_matrix = confusion_matrix(labels, preds)
print("Confusion Matrix:")
print(conf_matrix)

Confusion Matrix:
[[189   1   5  13   1   0  18   1   4   0]
 [  0 226   2   5   0   0   0   0   1   0]
 [  2   0 167   3  33   0  28   0   5   0]
 [  7   5   2 201   4   0   4   0   2   0]
 [  0   0  15   6 193   0  31   0   1   0]
 [  1   0   0   0   0 180   0  27   1  12]
 [ 24   1  27   7  13   0 162   0   7   0]
 [  0   0   0   0   0   3   0 246   0   7]
 [  1   0   1   0   0   0   4   0 244   0]
 [  0   0   0   0   0   3   0  15   0 239]]


In [8]:
print("Class labels:")
for i, name in enumerate(fashion_data.classes):
    print(f"{i}: {name}")

Class labels:
0: T-shirt/top
1: Trouser
2: Pullover
3: Dress
4: Coat
5: Sandal
6: Shirt
7: Sneaker
8: Bag
9: Ankle boot


The confusion matrix indicates that the model struggled most with the Pullover, Coat, Shirt, and T-shirt/top classes. The largest number of misclassifications occurred when Pullovers were classified as Coats (33 instances), followed by Coats classified as Shirts (31 instances) and Pullovers classified as Shirts (28 instances). These clothing items share similar visual characteristics, making them more difficult for the model to distinguish. In contrast, the model achieved strong performance on Trouser, Sneaker, Bag, and Ankle boot classes, which have more distinctive shapes and features.